# Cross-species RSA: steps 16.3 and 16.4
Use a **CPU** runtime and **Run all**. Mount Drive when prompted. This notebook automatically finds completed participant HDF5 results under **My Drive/rsa_colab/results_step16_2**. It handles both species directions and every dataset/model/class/radius combination found there.

Step 16.3 averages each source–target voxel pair equally across the available source participants. Step 16.4 writes 20 source-space threshold-count maps and a source-space mean-correlation map. Original distance maps, masks and configs are unnecessary.

Incomplete participant files are reported and excluded. The discovery cell prints the selected cohorts before computation. Group HDF5 files are saved in `<source>-group/`; NIfTI maps and exact threshold metadata are saved in `mean/`, beside participant folders. Re-running recomputes and replaces these derived outputs.


In [ ]:
%pip -q install numpy nibabel PyYAML h5py


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import importlib
import sys

DRIVE_ROOT = Path('/content/drive/MyDrive/rsa_colab')
INPUT_ROOT = DRIVE_ROOT / 'results_step16_2'
required = ['cross_species_rsa.py', 'discover_cross_species_groups.py']
missing = [str(DRIVE_ROOT / name) for name in required if not (DRIVE_ROOT / name).is_file()]
if missing:
    raise FileNotFoundError('Missing notebook support files: ' + ', '.join(missing))
sys.path.insert(0, str(DRIVE_ROOT))
importlib.invalidate_caches()
import cross_species_rsa as rsa
import discover_cross_species_groups as discovery
rsa = importlib.reload(rsa)
discovery = importlib.reload(discovery)
assert Path(rsa.__file__).resolve() == (DRIVE_ROOT / 'cross_species_rsa.py').resolve()
assert Path(discovery.__file__).resolve() == (DRIVE_ROOT / 'discover_cross_species_groups.py').resolve()
print('Input:', INPUT_ROOT)
print('Numerical module:', rsa.__file__)


## Discover available analyses
No path, radius or species settings need to be entered. By default every discovered analysis will run. To limit the run, edit `SELECTED` after inspecting the printed list.


In [ ]:
ANALYSES = discovery.discover_groups(INPUT_ROOT)
for index, analysis in enumerate(ANALYSES):
    print(f"[{index}] {analysis['dataset']} / {analysis['model']} / class_{analysis['rsa_class']} | "
          f"{analysis['specie']} r={analysis['radius']:g} -> {analysis['target_specie']} r={analysis['target_radius']:g}")
    print('  Source participants:', analysis['participants'])
    print('  Target participants:', analysis['target_participants'])
    print('  Masks:', analysis['mask_type'], '->', analysis['target_mask_type'])
SELECTED = ANALYSES  # optional subset, for example: [ANALYSES[0]]
SOURCE_BATCH, TARGET_BATCH = 128, 4096


## Step 16.3: group average matrices
Reads in bounded blocks. Undefined correlations propagate as NaN across source participants. Saved target cohorts, RDMs and voxel grids must agree.


In [ ]:
GROUP_OUTPUTS = []
for index, analysis in enumerate(SELECTED, 1):
    print(f"Step 16.3: analysis {index}/{len(SELECTED)}", flush=True)
    output = rsa.group_cross_similarity(**analysis, source_batch=SOURCE_BATCH,
                                       target_batch=TARGET_BATCH, verbose=True)
    GROUP_OUTPUTS.append(output)
    print('Saved group average:', output, flush=True)


## Step 16.4: threshold counts and mean maps
Thresholds are 20 evenly spaced values from the finite group minimum to maximum, including both endpoints. Each source voxel counts target voxels strictly above each threshold. Its mean uses finite target correlations only; an empty row has zero counts and a NaN mean.


In [ ]:
SUMMARY_OUTPUTS = []
for index, group_path in enumerate(GROUP_OUTPUTS, 1):
    print(f"Step 16.4: analysis {index}/{len(GROUP_OUTPUTS)}", flush=True)
    maps = rsa.summarize_group_cross_similarity(group_path, source_batch=SOURCE_BATCH,
                                               target_batch=TARGET_BATCH, verbose=True)
    SUMMARY_OUTPUTS.extend(maps)
    print('Saved 21 NIfTI maps and threshold JSON:', maps[0].parent, flush=True)
print(f'Done: {len(GROUP_OUTPUTS)} group matrices, {len(SUMMARY_OUTPUTS)} NIfTI maps.')
